In [ ]:
## Install Required Libraries

In [ ]:
## Create a virtual environment and install dependencies

#### Run this command in terminal
* python3 -m venv venv
* source venv/bin/activate
* pip install -r requirements.txt

In [ ]:
# install necessary libraries
!pip install tensorflow
!pip install stable-baselines3
!pip install gymnasium
!pip install matplotlib
!pip install pillow

## Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os


from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image

## Prepare Dataset

In [ ]:
import zipfile
import os

# Path to the directory containing the zip files
zip_dir = "/content" # dataset directory path

# List of zip files to extract
zip_files = [
    "trashnet-dataset.zip"     # dataset zip file name
]

for zf in zip_files:
    zip_path = os.path.join(zip_dir, zf)
    # The extraction directory is the zip_dir itself, which will create subfolders
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        print(f"Extracting {zf} to {zip_dir}/")
        zip_ref.extractall(zip_dir)
print("All dataset archives extracted.")

In [ ]:
print("Contents of trashnet dataset after extraction:")
!ls -F /content/trashnet-dataset

# Check contents of one of the extracted subfolders, e.g., 'metal'
print("\nContents of trashnet-dataset/metal:")
!ls -F /content/trashnet-dataset/metal | head -n 5 # Show first 5 items

## Load Dataset

In [ ]:
dataset_path = "/content/trashnet-dataset"

img_size = (224,224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

## Build Image Classification Model

In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

predictions = Dense(6, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Train Model (Fast Training)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

## Plot Training Results

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(["Train","Validation"])
plt.show()

## Image Prediction Function

In [ ]:
class_names = list(train_data.class_indices.keys())

def predict_waste(img_path):

    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)/255
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    class_id = np.argmax(prediction)

    return class_names[class_id]

## NLP Explanation System

This provides text explanation after classification.

In [ ]:
waste_explanations = {

"plastic":"Plastic should be recycled because it can be reused to produce new materials.",

"paper":"Paper waste should be recycled to reduce deforestation.",

"glass":"Glass can be recycled indefinitely without losing quality.",

"metal":"Metal waste can be melted and reused in manufacturing.",

"cardboard":"Cardboard should be flattened and recycled.",

"trash":"General waste should be disposed in landfill bins."

}

def generate_explanation(waste_type):

    return waste_explanations[waste_type]

## Reinforcement Learning Environment

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import random

class WasteEnv(gym.Env):

    def __init__(self):

        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Discrete(6)

    def reset(self, seed=None, options=None):

        self.state = random.randint(0,5)
        return self.state, {}

    def step(self, action):

        correct_action = 0

        reward = 1 if action == correct_action else -1

        terminated = True
        truncated = False

        return self.state, reward, terminated, truncated, {}

## Train RL Agent

In [ ]:
from stable_baselines3 import PPO

env = WasteEnv()

rl_model = PPO("MlpPolicy", env, verbose=0)

rl_model.learn(total_timesteps=2000)

## RL Bin Decision

In [ ]:
def choose_bin():

    obs,_ = env.reset()

    action,_ = rl_model.predict(obs)

    bins = ["Recycling Bin","Organic Bin","Landfill Bin"]

    return bins[action]

## Full AI Pipeline

In [ ]:
def smart_waste_assistant(image_path):

    waste_type = predict_waste(image_path)

    explanation = generate_explanation(waste_type)

    bin_decision = choose_bin()

    print("Predicted Waste Type:", waste_type)

    print("\nDisposal Explanation:")
    print(explanation)

    print("\nRecommended Bin:")
    print(bin_decision)

In [ ]:
## Test
smart_waste_assistant("/content/trashnet-dataset/metal/metal101.jpg")

### Expected Output

Predicted Waste Type: plastic

Disposal Explanation:
Plastic should be recycled because it can be reused to produce new materials.

Recommended Bin:
Recycling Bin